In [1]:
import pandas as pd

# 1. Carrega os arquivos CSV
df_items_pedido = pd.read_csv('../../Dados/olist_order_items_dataset.csv')
df_pedidos = pd.read_csv('../../Dados/olist_orders_dataset.csv')
df_pagamentos_pedido = pd.read_csv('../../Dados/olist_order_payments_dataset.csv')
df_revisoes_pedido = pd.read_csv('../../Dados/olist_order_reviews_dataset.csv')
df_produtos = pd.read_csv('../../Dados/olist_products_dataset.csv')

In [ ]:
# Problema: Alguns pedidos possuem mais de um vendedor, o que pode gerar inconsistências na análise de reviews. 
# Para garantir que cada pedido seja associado a apenas um vendedor, filtramos os pedidos que possuem apenas um vendedor.

df_items_pedido_filtrado = df_items_pedido.groupby('order_id').filter(lambda x: x['seller_id'].nunique() == 1)
df = df_items_pedido_filtrado.merge(df_revisoes_pedido.drop_duplicates(subset='order_id', keep='last'), how='left', on='order_id')
df = df.merge(df_pedidos, how='left', on='order_id')

# Calcular dias de atraso e diferença de data de entrega registrada pelo vendedor e cliente
# Se um pedido teve registro de entrega pelo cliente mas não pelo vendedor, subtrair pela média da diferença de data de entrega entre vendedor e cliente
# OBS: se tiver alguma data de entrega (vendedor ou cliente) é nula, o resultado será nulo
from datetime import datetime
df['dias_atraso'] = (pd.to_datetime(df['order_delivered_carrier_date']) - pd.to_datetime(df['order_estimated_delivery_date'])).dt.days
df['diferenca_data_entrega_vendedor_cliente'] = (pd.to_datetime(df['order_delivered_customer_date']) - pd.to_datetime(df['order_delivered_carrier_date'])).dt.days
media_diferenca = df['diferenca_data_entrega_vendedor_cliente'].filter(lambda x: x.notnull()).mean()
condicao = (df['dias_atraso'].isnull() & df['order_delivered_customer_date'].notnull())
df.loc[condicao, 'order_delivered_carrier_date'] = (pd.to_datetime(df.loc[condicao, 'order_delivered_customer_date']) - pd.Timedelta(days=media_diferenca.round())).dt.strftime('%Y-%m-%d %H:%M:%S')

# Remove pedidos que não possuem nem comentários nem registro de data de entrega, e remove avaliações duplicadas, evitando viés nos dados.
# Tratar avaliações repetidas, filtrando pela avaliação mais longa
df['review_comment'] = df['review_comment_title'].fillna("") + ': ' + df['review_comment_message'].fillna("")
df['review_char_length'] = df['review_comment'].str.len()
df = df.sort_values('review_char_length', ascending=False).drop_duplicates('order_id', keep='first')
df = df[(df['review_char_length'] > 2) | (df['order_delivered_carrier_date'].notna() | df['order_delivered_customer_date'].notna())]

df.to_csv('order_items_reviews_filtered.csv')
# Treinar uma RNN (LSTM no caso) para classificar comentários como: no prazo ou atrasado
# código...

# Embora os cálculos de dias de atraso estimem prazo, o cliente pode ter recebido antes ou depois, portanto, sobrescreveremos pela classificação da LSTM
def classify_review(row):
    return LSTM.predict(row['review_comment']) if row['review_char_length'] > 2 else row['status_entrega']
NO_PRAZO = 0
ATRASADO = 1
NAO_INFORMADO = 2
df['status_entrega'] = df.apply(lambda row: NO_PRAZO if row['dias_atraso'] <= 0 else ATRASADO if row['dias_atraso'] > 0 else NAO_INFORMADO, axis=1)
df['status_entrega'] = df.apply(classify_review, axis=1)


# Treinar uma ML (supervisionada) para identificar padrões prazo pela distancia entre vendedor e comprador 
# código

# Treinar 6 clusters (IA não supervisionada) para classificar prazo por regiões do brasil
# código